# Hugging Face — The Practical Toolkit for Transformers

Understanding transformer theory is important. But in practice, you almost never build
transformers from scratch. Instead, you use **Hugging Face** — the ecosystem that makes
state-of-the-art NLP accessible in a few lines of code.

Think of it as **sklearn for NLP**: pre-trained models, easy fine-tuning, and standardized APIs.

---
## Setup

Hugging Face requires installation. The core library is `transformers`.

In [ ]:
# !pip install transformers datasets torch accelerate

# If you need specific extras:
# !pip install transformers[torch]  # PyTorch backend
# !pip install sentencepiece         # For some tokenizers (T5, etc.)
# !pip install evaluate              # For metrics during training

---
## The Hugging Face Ecosystem

| Component | What It Does | Analogy |
|---|---|---|
| **transformers** | Pre-trained models + tokenizers | sklearn's estimators |
| **datasets** | Load & process datasets | sklearn's datasets module |
| **tokenizers** | Fast text → token ID conversion | sklearn's vectorizers |
| **evaluate** | Metrics (accuracy, F1, BLEU, etc.) | sklearn's metrics |
| **Hub** | 400K+ community models | Like PyPI for models |
| **Trainer** | Training loop with best practices | sklearn's fit() on steroids |

Everything connects through a consistent API: `Auto*` classes that figure out
the right model/tokenizer architecture automatically.

---
## Pipelines — The Fastest Way

Pipelines are the `predict()` of Hugging Face. One line: load a model and get predictions.
They handle tokenization, inference, and post-processing automatically.

In [ ]:
from transformers import pipeline

sentiment = pipeline('sentiment-analysis')

results = sentiment([
    "I absolutely loved this movie! Best film of the year.",
    "This was a complete waste of time. Terrible acting.",
    "The movie was okay, nothing special but not bad either.",
])

for text, result in zip(
    ["Loved it", "Terrible", "Okay"], results
):
    print(f"  {text:>10} → {result['label']} ({result['score']:.3f})")

In [ ]:
generator = pipeline('text-generation', model='gpt2')

result = generator(
    "The future of artificial intelligence is",
    max_new_tokens=50,
    num_return_sequences=1,
    temperature=0.7,
    do_sample=True,
)

print("Generated text:")
print(result[0]['generated_text'])

In [ ]:
summarizer = pipeline('summarization', model='facebook/bart-large-cnn')

article = """
Artificial intelligence has made remarkable progress in recent years, with large language
models demonstrating capabilities that were previously thought to be uniquely human.
These models can write essays, generate code, translate languages, and even engage in
complex reasoning tasks. However, significant challenges remain, including issues with
hallucination, bias, and the enormous computational resources required for training.
Researchers are actively working on making these models more efficient, more accurate,
and more aligned with human values. The field continues to evolve rapidly, with new
breakthroughs being announced almost weekly.
"""

summary = summarizer(article, max_length=60, min_length=20)
print("Summary:")
print(summary[0]['summary_text'])

In [ ]:
qa = pipeline('question-answering')

context = """
The transformer architecture was introduced in 2017 by Vaswani et al. in the paper
'Attention Is All You Need'. It replaced recurrent neural networks with self-attention
mechanisms, enabling parallel processing of sequences. BERT, released by Google in 2018,
was the first major model to use transformers for NLP understanding tasks.
"""

questions = [
    "When was the transformer architecture introduced?",
    "What did transformers replace?",
    "Who released BERT?",
]

for q in questions:
    answer = qa(question=q, context=context)
    print(f"Q: {q}")
    print(f"A: {answer['answer']} (confidence: {answer['score']:.3f})\n")

In [ ]:
classifier = pipeline('zero-shot-classification')

text = "The stock market crashed today, losing over 5% in a single session."
candidate_labels = ["finance", "sports", "technology", "politics", "entertainment"]

result = classifier(text, candidate_labels)

print(f"Text: {text}\n")
for label, score in zip(result['labels'], result['scores']):
    bar = '█' * int(score * 40)
    print(f"  {label:>15}: {score:.3f} {bar}")

print("\nZero-shot: no training needed! The model generalizes from its pre-training.")

---
## Tokenizers — How Text Becomes Numbers

Every model has its own tokenizer that converts text → token IDs. Modern tokenizers
use **subword** algorithms (BPE, WordPiece, SentencePiece) that split rare words
into known pieces.

```
"unhappiness" → ["un", "##happi", "##ness"]  (WordPiece)
"unhappiness" → ["un", "happiness"]           (BPE variant)
```

This handles rare words without a massive vocabulary.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

text = "Transformers revolutionized natural language processing!"
encoded = tokenizer(text, return_tensors='pt')

print(f"Original text: {text}")
print(f"\nToken IDs: {encoded['input_ids'][0].tolist()}")
print(f"Tokens:    {tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])}")
print(f"Attention mask: {encoded['attention_mask'][0].tolist()}")
print(f"\n[CLS] = classification token (start), [SEP] = separator token (end)")
print(f"Attention mask: 1 = real token, 0 = padding")

In [ ]:
words = ["cat", "unhappiness", "transformerization", "supercalifragilistic"]

print(f"{'Word':<25} {'Tokens':<40} {'# Tokens'}")
print("-" * 75)
for word in words:
    tokens = tokenizer.tokenize(word)
    print(f"{word:<25} {str(tokens):<40} {len(tokens)}")

print(f"\nCommon words → 1 token. Rare words → split into subwords (## prefix = continuation).")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

In [ ]:
gpt2_tok = AutoTokenizer.from_pretrained('gpt2')

text = "Hello world! How are you?"
bert_tokens = tokenizer.tokenize(text)
gpt2_tokens = gpt2_tok.tokenize(text)

print(f"Text: {text}\n")
print(f"BERT tokens:  {bert_tokens}")
print(f"GPT-2 tokens: {gpt2_tokens}")
print(f"\nDifferent models tokenize differently — always use the matching tokenizer!")

---
## Models — Under the Hood

The `Auto*` classes load the right architecture based on the model name:

| Class | Purpose |
|---|---|
| `AutoModel` | Base model — outputs hidden states |
| `AutoModelForSequenceClassification` | + classification head |
| `AutoModelForTokenClassification` | + per-token classification (NER) |
| `AutoModelForQuestionAnswering` | + QA head |
| `AutoModelForCausalLM` | + language model head (generation) |

In [ ]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained('bert-base-uncased')

text = "The cat sat on the mat"
inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

print(f"Input tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")
print(f"\nLast hidden state shape: {outputs.last_hidden_state.shape}")
print(f"  → (batch_size=1, seq_len=8, hidden_dim=768)")
print(f"\nEach token now has a 768-dim context-aware embedding.")
print(f"Unlike Word2Vec, 'mat' here is different from 'mat' in 'yoga mat'.")

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"BERT-base parameters: {total_params:,}")
print(f"\nModel architecture (first few layers):")
for name, param in list(model.named_parameters())[:6]:
    print(f"  {name}: {param.shape}")
print("  ...")

---
## Fine-Tuning a Text Classifier

The real power: take a pre-trained model (trained on billions of words) and
adapt it to YOUR specific task with a small labeled dataset.

```
Pre-trained BERT → add classification head → train on your data → done
```

The `Trainer` API handles the training loop, evaluation, and checkpointing.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np

train_texts = [
    "This product is amazing and works perfectly",
    "Love it! Best purchase I've made",
    "Excellent quality and fast shipping",
    "Highly recommend this to everyone",
    "Great value for the price",
    "Wonderful experience, will buy again",
    "Terrible product, broke after one day",
    "Worst purchase ever, do not buy",
    "Awful quality, complete waste of money",
    "Horrible customer service and bad product",
    "Very disappointed, returning immediately",
    "Poor quality, not as described at all",
]
train_labels = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

eval_texts = [
    "This is fantastic, I love it",
    "Terrible, would not recommend",
    "Great product, fast delivery",
    "Broken on arrival, very bad",
]
eval_labels = [1, 0, 1, 0]

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=64)

train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels}).map(tokenize, batched=True)
eval_dataset = Dataset.from_dict({'text': eval_texts, 'label': eval_labels}).map(tokenize, batched=True)

print(f"Train: {len(train_dataset)} examples")
print(f"Eval:  {len(eval_dataset)} examples")
print(f"\nTokenized sample:")
print(f"  Text: {train_texts[0]}")
print(f"  Token IDs: {train_dataset[0]['input_ids'][:15]}...")
print(f"  Label: {train_dataset[0]['label']}")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy='epoch',
    logging_steps=5,
    save_strategy='no',
    report_to='none',
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {'accuracy': accuracy}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

print("Training... (this fine-tunes ALL of DistilBERT + the classification head)")
trainer.train()

In [ ]:
test_texts = [
    "Amazing product, exceeded my expectations!",
    "Garbage quality, total ripoff",
    "Pretty decent for the price",
]

fine_tuned_pipeline = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)
results = fine_tuned_pipeline(test_texts)

label_map = {'LABEL_0': 'Negative', 'LABEL_1': 'Positive'}
for text, result in zip(test_texts, results):
    label = label_map.get(result['label'], result['label'])
    print(f"  {label:>8} ({result['score']:.3f}) | {text}")

---
## Text Generation with GPT-2

GPT-2 generates text autoregressively — one token at a time.
Key parameters that control the output:

| Parameter | What It Does | Low Value | High Value |
|---|---|---|---|
| `temperature` | Controls randomness | Deterministic, repetitive | Creative, chaotic |
| `top_k` | Only consider top k tokens | Conservative | More variety |
| `top_p` (nucleus) | Consider tokens summing to probability p | Focused | Diverse |

In [ ]:
generator = pipeline('text-generation', model='gpt2')

prompt = "In the year 2050, artificial intelligence"

configs = [
    {'temperature': 0.1, 'label': 'Very low temp (conservative)'},
    {'temperature': 0.7, 'label': 'Medium temp (balanced)'},
    {'temperature': 1.5, 'label': 'High temp (creative/chaotic)'},
]

for config in configs:
    result = generator(
        prompt,
        max_new_tokens=40,
        temperature=config['temperature'],
        do_sample=True,
        top_k=50,
        num_return_sequences=1,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    print(f"\n{config['label']}:")
    print(f"  {result[0]['generated_text']}")

---
## Named Entity Recognition (NER)

Extract entities (people, organizations, locations, dates) from text.

In [ ]:
ner = pipeline('ner', aggregation_strategy='simple')

text = "Elon Musk founded SpaceX in 2002 in Hawthorne, California. The company launched Falcon 9 from Cape Canaveral."

entities = ner(text)

print(f"Text: {text}\n")
print(f"{'Entity':<25} {'Type':<10} {'Score':<8}")
print("-" * 45)
for entity in entities:
    print(f"{entity['word']:<25} {entity['entity_group']:<10} {entity['score']:.3f}")

---
## The Model Hub

The Hugging Face Hub hosts 400,000+ models. You can filter by:
- **Task**: text-classification, question-answering, summarization, ...
- **Language**: English, multilingual, Chinese, ...
- **Size**: from tiny (66M) to massive (70B+)
- **Framework**: PyTorch, TensorFlow, JAX

Popular model families:

| Family | Type | Good For |
|---|---|---|
| `bert-base-uncased` | Encoder | Classification, NER |
| `distilbert-base-uncased` | Encoder (smaller) | Same, but 60% faster |
| `roberta-base` | Encoder (improved BERT) | Better classification |
| `gpt2` | Decoder | Text generation |
| `facebook/bart-large-cnn` | Enc-Dec | Summarization |
| `t5-base` | Enc-Dec | Any text-to-text task |
| `sentence-transformers/*` | Encoder | Semantic search, embeddings |

Using any model is the same pattern:
```python
tokenizer = AutoTokenizer.from_pretrained('model-name')
model = AutoModelFor*.from_pretrained('model-name')
```

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

tok = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
emb_model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

def get_embedding(text):
    inputs = tok(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        output = emb_model(**inputs)
    return output.last_hidden_state[:, 0, :]  # CLS token

sentences = [
    "The cat sat on the mat",
    "A kitten rested on the rug",
    "Stock prices rose sharply today",
]

embeddings = torch.cat([get_embedding(s) for s in sentences])

from torch.nn.functional import cosine_similarity

print("Semantic similarity (transformer embeddings):\n")
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0))
        print(f"  '{sentences[i][:30]}' vs '{sentences[j][:30]}'")
        print(f"  Similarity: {sim.item():.3f}\n")

print("Transformer embeddings capture semantic meaning — similar sentences are close!")

---

## Summary

**The Hugging Face pattern:**

```python
# Quick inference
pipe = pipeline('task-name', model='model-name')
result = pipe("your text here")

# Fine-grained control
tokenizer = AutoTokenizer.from_pretrained('model-name')
model = AutoModelFor*.from_pretrained('model-name')
inputs = tokenizer(text, return_tensors='pt')
outputs = model(**inputs)

# Fine-tuning
trainer = Trainer(model=model, args=args, train_dataset=ds)
trainer.train()
```

**Key takeaways:**
- Pipelines for quick experiments (sentiment, generation, NER, QA)
- AutoTokenizer + AutoModel for control over inputs/outputs
- Trainer API for fine-tuning with minimal boilerplate
- The Hub has a model for almost every NLP task

**Next notebook:** LLMs and RAG — using these models at massive scale with your own data.